In [1]:
import os
import json
import xml.etree.ElementTree as ET

import epo_ops
import spacy
from dotenv import load_dotenv
from PatentProvider import PatentProvider
from kg.formatting.formatting_manager import FormattingManager



c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
def get_last_id(filename: str) -> int:
    if not os.path.exists(filename):
        return 0  ##working 
    last_id = 0
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue  # skip malformed liness

            if isinstance(obj, dict) and "id" in obj and isinstance(obj["id"], int):
                last_id = max(last_id, obj["id"])

    return last_id

In [6]:
nlp = spacy.load("en_core_web_trf")

def iterateSeveralPatentsDescription(base_sentences):
    patents = [1000000,1502502,1502503,1502504,1502507,1502509,1502510,1502513,1502516,3115868,3483653]
    for patentId in patents:
        description_text = PatentProvider().getDescription(patentId)
        doc = nlp(description_text)
        base_sentences.extend([sent.text.strip() for sent in doc.sents if sent.text.strip()])

In [7]:
import os
print(os.cpu_count())


24


In [8]:
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

base_sentences = []
iterateSeveralPatentsDescription(base_sentences)

total = len(base_sentences)
print(f"Total base sentences to split: {total}")

formatting_manager = FormattingManager()

def split_one(base: str):
    return formatting_manager.split(base)

MAX_WORKERS = 24
all_entries = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(split_one, b): i for i, b in enumerate(base_sentences)}

    done_count = 0
    for future in as_completed(futures):
        idx = futures[future]  # original index (for debugging/logging)
        try:
            sub = future.result()
            done_count += 1
            percent = done_count / total * 100
            print(f"[{done_count}/{total}] {percent:.1f}% done (base idx {idx})")
            for s in sub:
                all_entries.append({"data": {"text": s.text}})
        except Exception as e:
            print(f"Error on {idx}: {e}")


obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
obP7iHGVo6HMzenhj3uXdl8T7E7zGPpVGhcRDthgmf6rzZmd
Total base sentences to split: 4366
[1/4366] 0.0% done (base idx 6)
[2/4366] 0.0% done (base idx 21)
[3/4366] 0.1% done (base idx 9)
[4/4366] 0.1% done (base idx 23)
[5/4366] 0.1% done (base idx 15)
[6/4366] 0.1% done (base idx 1)
[7/4366] 0.2% done (base idx 3)
[8/4366] 0.2% done (base idx 4)
[9/4366] 0.2% done (base idx 16)
[10/4366] 0.2% done (base idx 17)
[11/4366] 0.3% done (base idx 10)
[12/4366] 0.3% done (base idx 19)
[13/4366] 0.3% done (base idx 1

In [9]:
output_file = "training/descriptions_patents_splitted.json"
entries = []
current_id = 0

for entry in all_entries:
    current_id += 1
    entry["id"] = current_id
    entries.append(entry)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(entries)} items to {output_file}.")
print(f"Last assigned ID: {current_id}")



Saved 13003 items to training/descriptions_patents_splitted.json.
Last assigned ID: 13003


In [ ]:
import json

INPUT_EXPORT = "training/descriptions_patents_splitted_labeled_IFO.json"
OUTPUT_DATASET = "training/model_info/sentence_classification_dataset_training.jsonl"

label_to_id = {
    "HIGH_INFO": 0,
    "LOW_INFO": 1,
    "FIGURE_RELATED": 2,
    "UNINFORMATIVE": 3
}

with open(INPUT_EXPORT, "r", encoding="utf-8") as f:
    tasks = json.load(f)

with open(OUTPUT_DATASET, "w", encoding="utf-8") as out:
    for task in tasks:
        text = task["data"].get("text", "").strip()
        if not text:
            continue

        anns = task.get("annotations", [])
        if not anns:
            continue

        # only use the first completed annotation
        results = anns[0].get("result", [])
        if not results:
            continue

        value = results[0].get("value", {})
        choices = value.get("choices", [])
        if not choices:
            continue

        label_name = choices[0]
        if label_name not in label_to_id:
            continue

        out.write(json.dumps({
            "id": task.get("id"),
            "text": text,
            "label_name": label_name,
            "label": label_to_id[label_name]
        }, ensure_ascii=False) + "\n")

print("Dataset written to", OUTPUT_DATASET)


Wrote cleaned dataset to sentence_classification_dataset_for_training_post_labelstudio.jsonl


In [ ]:
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


# -------------------------
# 1) Load JSONL dataset
# -------------------------
# Must be a JSON Lines file with fields: text, label
data_files = {
    "train": "training/model_info/sentence_classification_dataset_training.jsonl",
    # later you can add "validation": "val.jsonl"
}
raw_datasets = load_dataset("json", data_files=data_files, split="train")

# If you want to create a small validation split from train:
raw_datasets = raw_datasets.train_test_split(test_size=0.2, seed=42)
train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets["test"]

# Infer label space from data
labels = sorted(set(train_dataset["label"]))
num_labels = len(labels)
print("Detected labels:", labels)

# Optional: name mapping for convenience
id2label = {
   "HIGH_INFO": 0,
    "LOW_INFO": 1,
    "FIGURE_RELATED": 2,
    "UNINFORMATIVE": 3
}
label2id = {v: k for k, v in id2label.items() if k in labels}

print("id2label:", id2label)
print("label2id:", label2id)


# -------------------------
# 2) Tokenizer & model
# -------------------------
model_name = "anferico/bert-for-patents"  # good small baseline

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
    )

encoded_train = train_dataset.map(preprocess, batched=True)
encoded_eval = eval_dataset.map(preprocess, batched=True)

encoded_train = encoded_train.rename_column("label", "labels")
encoded_eval = encoded_eval.rename_column("label", "labels")

encoded_train.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
encoded_eval.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


# -------------------------
# 3) Data collator & metrics
# -------------------------
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# -------------------------
# 4) TrainingArguments & Trainer
# -------------------------
training_args = TrainingArguments(
    output_dir="./sentence_classifier_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save final model
trainer.save_model("training/model_info/done/sentence_classifier_model")
tokenizer.save_pretrained("training/model_info/done/sentence_classifier_model")
print("Model saved to ./sentence_classifier_model")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json

model_path = "./sentence_classifier_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

id2label = model.config.id2label

def classify_sentence(text: str):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)[0]
    pred_id = int(torch.argmax(probs))
    return id2label[pred_id], probs.tolist()

# Example: filter sentences.txt
informative_sentences = []
with open("sentences.txt", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        text = obj["data"]["text"]
        label_name, probs = classify_sentence(text)
        if label_name in ("INFORMATIVE", "FIGURE_RELATED"):
            informative_sentences.append(obj)

print("Kept", len(informative_sentences), "sentences as informative/figure-related.")
